# Module 1: Agentic RAG Pipeline for Financial Analysis

## Learning Objectives
By the end of this module, you will understand:

1. ✅ **The Problem**: LLMs have knowledge cutoff; they can't answer questions about recent data
2. ✅ **The Solution**: RAG (Retrieval-Augmented Generation) - provide context to the LLM
3. ✅ **Retrieval**: How to search for relevant information using `minsearch`
4. ✅ **Prompt Building**: How to combine context + question into an effective prompt
5. ✅ **Generation**: How to call the LLM with the context
6. ✅ **Class-Based Design**: How to organize this into reusable classes

---

## Journey
❌ Without RAG → ✅ With RAG → 🚀 Class-Based RAG Pipeline

## Setup: Dependencies and Configuration

In [1]:
# Check Python version
import sys
print(f"✅ Python version: {sys.version}")

✅ Python version: 3.11.15 (main, Mar  3 2026, 00:52:57) [Clang 17.0.0 (clang-1700.6.3.2)]


In [2]:
# Load environment variables (API keys)
from dotenv import load_dotenv
import os

load_dotenv()
key_present = bool(os.getenv("OPENAI_API_KEY"))

if key_present:
    print("✅ OPENAI_API_KEY loaded successfully")
else:
    print("❌ ERROR: OPENAI_API_KEY not found. Please set it in .env file")

✅ OPENAI_API_KEY loaded successfully


---

# Part 1: The Problem - LLMs Without Context

## What happens when you ask an LLM about recent data?

LLMs like GPT-4 were trained on data up to a certain date (usually 2023 or earlier). They don't know about recent events, earnings, or data unless you tell them.

Let's see what happens when we ask without providing context:

In [3]:
from openai import OpenAI

client = OpenAI()

# Ask about recent financial data WITHOUT providing context
question = "What was Microsoft's gross margin in fiscal Q3 2026?"

print("❌ PROBLEM: Asking WITHOUT context")
print(f"Question: {question}")
print("\n" + "="*60)

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": question}],
)

print("\nLLM Response:")
print(response.choices[0].message.content)
print("\n" + "="*60)
print("Notice: The LLM can't answer! It doesn't know about Q3 2026 data.")

❌ PROBLEM: Asking WITHOUT context
Question: What was Microsoft's gross margin in fiscal Q3 2026?


LLM Response:
I'm sorry, but I cannot provide real-time or future financial data, including Microsoft's gross margin for fiscal Q3 2026, as my training only includes information up to October 2023. For the most accurate and updated financial statistics, please refer to Microsoft's official financial reports or trusted financial news sources.

Notice: The LLM can't answer! It doesn't know about Q3 2026 data.


### Why This Happens
- LLMs have a **knowledge cutoff** (training data ends in early 2024)
- They can't access real-time data or recent filings
- They refuse to guess because accuracy matters for financial data
- **Solution**: Provide the data to the LLM (context)

---

# Part 2: The Solution - Adding Context (RAG)

## RAG = Retrieval-Augmented Generation

The idea is simple:
1. **Retrieve**: Find relevant information
2. **Augment**: Add it to the prompt
3. **Generate**: Let the LLM answer based on that context

Let's try with context:

### Step 1: Load Financial Data (Microsoft's MD&A)

In [4]:
# This is real text from Microsoft's 10-Q filing (MD&A section)
mdna_excerpt = """
Microsoft is a technology company committed to making digital technology and artificial 
intelligence ("AI") available broadly and doing so responsibly, with a mission to empower 
every person and every organization on the planet to achieve more.

Highlights from the third quarter of fiscal year 2026 compared with the third quarter of 
fiscal year 2025 included:
- Microsoft Cloud revenue increased 29% to $54.5 billion.
- Commercial remaining performance obligation increased 99% to $627 billion.
- Microsoft 365 Commercial cloud revenue increased 19%.
- Microsoft 365 Consumer cloud revenue increased 33%.
- LinkedIn revenue increased 12%.
- Dynamics 365 revenue increased 22%.
- Azure and other cloud services revenue increased 40%.
- Windows OEM and Devices revenue decreased 2%.
- Xbox content and services revenue decreased 5%.
- Search advertising revenue excluding traffic acquisition costs increased 12%.

Industry Trends and Opportunities:
We have a long-term strategic partnership with OpenAI which was originally established in 
2019. In October 2025 and April 2026, we extended this partnership and continue to build on 
our shared vision to advance artificial intelligence responsibly and make its benefits broadly 
accessible.

Economic Conditions, Challenges, and Risks:
The markets for software, devices, and cloud-based services are dynamic and highly 
competitive. Our competitors are developing new software and devices, while also deploying 
competing cloud-based services for consumers and businesses.

The investments we are making in cloud and AI infrastructure and devices will continue to 
increase our operating costs and may decrease our operating margins. We continue to identify 
and evaluate opportunities to expand our datacenter locations and increase our server 
capacity to meet the evolving needs of our customers, particularly given the growing demand 
for AI services.
"""

print("✅ Loaded Microsoft's MD&A from 10-Q filing")
print(f"Data length: {len(mdna_excerpt)} characters")

✅ Loaded Microsoft's MD&A from 10-Q filing
Data length: 1910 characters


### Step 2: Create a Prompt with Context

In [5]:
# Same question, but WITH context
question = "What were the main drivers of Microsoft's revenue growth?"

# Create a prompt that includes both context and question
prompt_with_context = f"""You are a financial data expert. Answer the QUESTION using 
ONLY the information in CONTEXT below. If the context does not contain enough
information to answer, say "I don't have enough information in the provided filings."

CONTEXT:
{mdna_excerpt}

QUESTION: {question}

ANSWER:"""

print("✅ SOLUTION: Including context in the prompt")
print(f"Question: {question}")
print("\n" + "="*60)
print("Calling LLM with context...")
print("="*60 + "\n")

✅ SOLUTION: Including context in the prompt
Question: What were the main drivers of Microsoft's revenue growth?

Calling LLM with context...



In [6]:
# Call the LLM WITH context
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{"role": "user", "content": prompt_with_context}],
)

print("✅ LLM Response (with context):")
print(response.choices[0].message.content)
print("\n" + "="*60)
print("Success! The LLM provided an accurate answer using the context we provided.")

✅ LLM Response (with context):
The main drivers of Microsoft's revenue growth included a significant increase in Microsoft Cloud revenue, which rose 29% to $54.5 billion, as well as substantial growth in various segments such as Azure and other cloud services revenue, which increased 40%. Additionally, Microsoft 365 Commercial and Consumer cloud revenues grew by 19% and 33%, respectively. Other contributing factors included a 12% increase in LinkedIn revenue and a 22% increase in Dynamics 365 revenue.

Success! The LLM provided an accurate answer using the context we provided.


---

# Part 3: Scaling RAG - Multiple Documents and Search

## The Problem with the Previous Approach
- ❌ We sent the ENTIRE `mdna_excerpt` every time
- ❌ Not efficient if we have 100 documents
- ❌ LLM has to read irrelevant information
- ❌ Wastes tokens (costs money, takes time)

## The Better Way: Search for Relevant Documents
- ✅ Store documents in a searchable index
- ✅ Search for only relevant chunks
- ✅ Send only matching documents to LLM
- ✅ More efficient, scales to 1000s of documents

Let's build this:

### Step 1: Create Sample Documents (Multiple Companies)

In [7]:
# Sample financial documents from 3 companies (MSFT, AAPL, GOOGL)
# Each with metadata: ticker, form_type, date, period, text

documents = [
    {
        "ticker": "MSFT",
        "form_type": "10-Q",
        "filed_date": "2024-08-01",
        "fiscal_period": "2024-Q4",
        "text": "Gross margin improved to 69.8% from 68.5% in the prior year quarter. The improvement was driven by favorable product mix in cloud services and higher Azure utilization rates. Operating expenses increased 12% year-over-year due to investments in AI research and infrastructure expansion.",
    },
    {
        "ticker": "AAPL",
        "form_type": "10-Q",
        "filed_date": "2024-08-15",
        "fiscal_period": "2024-Q3",
        "text": "Revenue increased 5% to $85.8 billion driven by strong iPhone 16 Pro demand and services growth. Gross margin declined slightly to 46.2% from 46.8% due to pricing pressure in India and China markets. Operating margin remained stable at 31% despite supply chain inflation.",
    },
    {
        "ticker": "GOOGL",
        "form_type": "10-Q",
        "filed_date": "2024-08-10",
        "fiscal_period": "2024-Q2",
        "text": "Gross margin expanded to 57.3% from 55.9% primarily due to improved efficiency in data center operations and higher-margin cloud sales. The company's AI-related investments drove a 22% increase in R&D expenses year-over-year. Traffic acquisition costs rose 8% reflecting competitive pressure in search.",
    },
    {
        "ticker": "MSFT",
        "form_type": "10-Q",
        "filed_date": "2024-05-15",
        "fiscal_period": "2024-Q3",
        "text": "Operating margin improved to 52% from 50.5% in the prior year, benefiting from operating leverage in cloud services. Azure and other cloud revenue grew 29% year-over-year, offsetting slower growth in legacy productivity software. Gross margin remained stable as product mix shifts balanced pricing gains.",
    },
    {
        "ticker": "AAPL",
        "form_type": "10-Q",
        "filed_date": "2024-05-01",
        "fiscal_period": "2024-Q2",
        "text": "Gross margin contracted to 45.8% from 47.2% as foreign exchange headwinds and manufacturing costs in Taiwan increased. iPhone sales growth slowed to 2% amid market saturation in developed countries. Services revenue grew 16% providing higher-margin recurring revenue.",
    },
]

print(f"✅ Created {len(documents)} sample financial documents")
for i, doc in enumerate(documents, 1):
    print(f"  {i}. {doc['ticker']} - {doc['fiscal_period']}")

✅ Created 5 sample financial documents
  1. MSFT - 2024-Q4
  2. AAPL - 2024-Q3
  3. GOOGL - 2024-Q2
  4. MSFT - 2024-Q3
  5. AAPL - 2024-Q2


### Step 2: Build a Searchable Index

In [8]:
from minsearch import Index

# Create an index with searchable fields
index = Index(
    text_fields=["text"],  # Search in the text field
    keyword_fields=["ticker", "form_type", "fiscal_period"],  # Filter by these
)

# Build the index with our documents
index.fit(documents)

print("✅ Created searchable index")
print(f"   Text fields: ['text']")
print(f"   Keyword fields: ['ticker', 'form_type', 'fiscal_period']")

✅ Created searchable index
   Text fields: ['text']
   Keyword fields: ['ticker', 'form_type', 'fiscal_period']


### Step 3: Search for Relevant Documents

In [9]:
# Search for documents about Microsoft's gross margin
query = "gross margin improvement"
filter_ticker = {"ticker": "MSFT"}  # Only search Microsoft documents
num_results = 2  # Return top 2 matches

print(f"🔍 Searching for: '{query}'")
print(f"   Filter: {filter_ticker}")
print(f"   Top results: {num_results}")
print("\n" + "="*60 + "\n")

search_results = index.search(
    query=query,
    filter_dict=filter_ticker,
    num_results=num_results,
)

print(f"✅ Found {len(search_results)} matching documents:\n")
for i, result in enumerate(search_results, 1):
    print(f"Result {i}:")
    print(f"  Company: {result['ticker']}")
    print(f"  Period: {result['fiscal_period']}")
    print(f"  Text: {result['text'][:100]}...")
    print()

🔍 Searching for: 'gross margin improvement'
   Filter: {'ticker': 'MSFT'}
   Top results: 2


✅ Found 2 matching documents:

Result 1:
  Company: MSFT
  Period: 2024-Q4
  Text: Gross margin improved to 69.8% from 68.5% in the prior year quarter. The improvement was driven by f...

Result 2:
  Company: MSFT
  Period: 2024-Q3
  Text: Operating margin improved to 52% from 50.5% in the prior year, benefiting from operating leverage in...



---

# Part 4: Building Reusable Components

## Lesson 5: Prompt Building and Formatting

Notice that we keep doing the same steps:
1. Search for documents
2. Join them into a context string
3. Format into a prompt
4. Call the LLM

Instead of repeating this, let's make reusable functions:

In [10]:
# Define a prompt template (can be reused)
PROMPT_TEMPLATE = """You are a financial data expert. Answer the QUESTION using
ONLY the information in CONTEXT below. If the context does not contain enough
information to answer, say "I don't have enough information in the provided
filings to answer that" — do not use outside knowledge or guess a number.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

def build_prompt(question: str, search_results: list[dict]) -> str:
    """
    Combine search results into a prompt.
    
    Args:
        question: The user's question
        search_results: List of dicts with 'text' field
    
    Returns:
        Formatted prompt string
    """
    # Join all texts with double newlines for readability
    context = "\n\n".join(r["text"] for r in search_results)
    # Replace placeholders in template
    return PROMPT_TEMPLATE.format(context=context, question=question)

def call_llm(prompt: str, model: str = "gpt-4o-mini", temperature: float = 0.0) -> str:
    """
    Call OpenAI API with a prompt.
    
    Args:
        prompt: The formatted prompt
        model: Which model to use
        temperature: Creativity level (0=deterministic, 1=creative)
    
    Returns:
        LLM's response
    """
    response = client.chat.completions.create(
        model=model,
        temperature=temperature,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content

print("✅ Created reusable functions:")
print("   - build_prompt(question, search_results)")
print("   - call_llm(prompt, model, temperature)")

✅ Created reusable functions:
   - build_prompt(question, search_results)
   - call_llm(prompt, model, temperature)


### Using These Functions Together

In [11]:
# Now the workflow is clean and simple

question = "What was Microsoft's gross margin improvement in Q4 2024?"

print(f"❓ Question: {question}\n")

# Step 1: Search
results = index.search(
    query=question,
    filter_dict={"ticker": "MSFT"},
    num_results=2,
)
print(f"✅ Step 1 - Search: Found {len(results)} documents")

# Step 2: Build prompt
prompt = build_prompt(question, results)
print(f"✅ Step 2 - Build: Created formatted prompt")

# Step 3: Call LLM
answer = call_llm(prompt)
print(f"✅ Step 3 - Generate: Got answer from LLM")

print("\n" + "="*60)
print(f"\n📋 Generated Prompt:")
print(prompt)

print("\n" + "="*60)
print(f"\n💡 LLM Answer:")
print(answer)

❓ Question: What was Microsoft's gross margin improvement in Q4 2024?

✅ Step 1 - Search: Found 2 documents
✅ Step 2 - Build: Created formatted prompt
✅ Step 3 - Generate: Got answer from LLM


📋 Generated Prompt:
You are a financial data expert. Answer the QUESTION using
ONLY the information in CONTEXT below. If the context does not contain enough
information to answer, say "I don't have enough information in the provided
filings to answer that" — do not use outside knowledge or guess a number.

CONTEXT:
Gross margin improved to 69.8% from 68.5% in the prior year quarter. The improvement was driven by favorable product mix in cloud services and higher Azure utilization rates. Operating expenses increased 12% year-over-year due to investments in AI research and infrastructure expansion.

Operating margin improved to 52% from 50.5% in the prior year, benefiting from operating leverage in cloud services. Azure and other cloud revenue grew 29% year-over-year, offsetting slower growth in l

---

# Part 5: Class-Based Refactoring

## The Problem with Functions
Our current approach works, but has issues:
- ❌ Code is scattered across cells
- ❌ Have to recreate `index`, `client` each time
- ❌ Hard to reuse and share
- ❌ Difficult to test

## The Solution: Class-Based Design
- ✅ Everything in organized classes
- ✅ Initialize once, use many times
- ✅ Easy to extend and modify
- ✅ Production-ready

Let's use the class-based `RAGPipeline`:

In [12]:
from rag_pipeline import RAGPipeline

# Create ONE pipeline instance
rag = RAGPipeline(
    documents=documents,
    llm_model="gpt-4o-mini",
    llm_temperature=0.0,
)

print("✅ RAGPipeline initialized")
print("   - Index built")
print("   - LLM client ready")
print("   - Prompt builder configured")
print("\n🎉 Now ready to answer questions!")

✅ RAGPipeline initialized
   - Index built
   - LLM client ready
   - Prompt builder configured

🎉 Now ready to answer questions!


### Example 1: Simple Answer

In [13]:
# Just one line to get an answer!
question_1 = "What was Microsoft's gross margin improvement in Q4 2024?"

answer_1 = rag.answer(
    question=question_1,
    filter_dict={"ticker": "MSFT"},
    num_results=2,
)

print(f"❓ Q: {question_1}")
print(f"\n✅ A: {answer_1}")

❓ Q: What was Microsoft's gross margin improvement in Q4 2024?

✅ A: Microsoft's gross margin improvement in Q4 2024 was from 68.5% in the prior year quarter to 69.8%.


### Example 2: Answer with Context

In [14]:
# Get both the answer AND the context used
question_2 = "What were Apple's revenue trends?"

result = rag.answer_with_context(
    question=question_2,
    filter_dict={"ticker": "AAPL"},
    num_results=2,
)

print(f"❓ Question: {result['question']}")
print(f"\n✅ Answer: {result['answer']}")
print(f"\n📚 Sources: {len(result['context'])} documents")
for i, doc in enumerate(result['context'], 1):
    print(f"   {i}. {doc['ticker']} - {doc['fiscal_period']}")

❓ Question: What were Apple's revenue trends?

✅ Answer: Apple's revenue increased by 5% to $85.8 billion, driven by strong demand for the iPhone 16 Pro and growth in services revenue, which grew by 16%.

📚 Sources: 2 documents
   1. AAPL - 2024-Q2
   2. AAPL - 2024-Q3


### Example 3: Cross-Company Analysis

In [15]:
# Search without filtering by company
question_3 = "Which companies increased their gross margins?"

answer_3 = rag.answer(
    question=question_3,
    filter_dict=None,  # No filter = search ALL companies
    num_results=3,
)

print(f"❓ Q: {question_3}")
print(f"\n✅ A: {answer_3}")

❓ Q: Which companies increased their gross margins?

✅ A: I don't have enough information in the provided filings to answer that.


---

# Summary: The Complete RAG Journey

## What We Learned

| Stage | What | Why |
|-------|------|-----|
| **Problem** | ❌ LLM without context | Knowledge cutoff; can't answer recent questions |
| **Solution** | ✅ Add context to prompt | LLM can now answer using provided data |
| **Scaling** | 🔍 Search for relevant docs | Don't send everything; be efficient |
| **Reusability** | 🔧 Reusable functions | Don't repeat the same code |
| **Production** | 🚀 Class-based design | Professional, tested, extensible |

## The RAG Pipeline Flow

```
User Question
    ↓
Search (Find relevant documents)
    ↓
Retrieve (Get the matching chunks)
    ↓
Build Prompt (Combine context + question)
    ↓
Call LLM (Send to OpenAI)
    ↓
Answer (Return to user)
```

## Key Benefits

✅ **Accurate**: Based on real data, not hallucinations
✅ **Current**: Works with recent documents
✅ **Efficient**: Only sends relevant information
✅ **Scalable**: Works with thousands of documents
✅ **Professional**: Class-based, extensible design

## Next Steps

1. **Review**: `rag_pipeline.py` to understand the implementation
2. **Read**: `CLASS_REFACTOR_GUIDE.md` for architecture details
3. **Experiment**: Try different questions and documents
4. **Extend**: Add new features (caching, different LLMs, etc.)
5. **Apply**: Use in your own financial analysis tasks

---

## 🎯 Checkpoint: Do You Understand?

- [ ] Why RAG is needed (LLMs have knowledge cutoff)
- [ ] The 3 steps: Retrieve → Augment → Generate
- [ ] How to search documents efficiently
- [ ] How to build and format prompts
- [ ] Why class-based design is better
- [ ] How to use RAGPipeline

If you checked them all, you've mastered Module 1! 🎉